## 🌾 Machine Learning: Crop Recommendation System

### 📚 Deskripsi Dataset

Pada studi kasus ini, kita akan membuat **sistem rekomendasi tanaman** berbasis **Machine Learning** yang bertujuan membantu petani dalam menentukan jenis tanaman terbaik berdasarkan kondisi lingkungan dan tanah.

Dataset yang digunakan berisi **data agrikultur** dengan berbagai parameter penting yang mempengaruhi pertumbuhan tanaman. Data ini mencakup **unsur hara tanah (N, P, K)**, **suhu**, **kelembaban udara**, **tingkat keasaman tanah (pH)**, dan **curah hujan**. Berdasarkan parameter-parameter ini, sistem akan memberikan **rekomendasi jenis tanaman** yang paling sesuai.

Berikut adalah deskripsi dari setiap kolom dalam dataset:

| **Kolom**        | **Tipe Data**          | **Deskripsi**                                                                                     |
|------------------|------------------------|---------------------------------------------------------------------------------------------------|
| **N**            | `int`                  | Kandungan Nitrogen dalam tanah, diukur dalam satuan mg/kg.                                       |
| **P**            | `int`                  | Kandungan Phosphorus dalam tanah, diukur dalam satuan mg/kg.                                     |
| **K**            | `int`                  | Kandungan Potassium dalam tanah, diukur dalam satuan mg/kg.                                      |
| **temperature**  | `float`                | Suhu lingkungan tempat tanaman tumbuh, diukur dalam derajat Celcius (°C).                       |
| **humidity**     | `float`                | Kelembaban udara di lingkungan tumbuh, diukur dalam persen (%).                                  |
| **ph**           | `float`                | Tingkat keasaman tanah (pH), menunjukkan kondisi asam atau basa pada tanah.                      |
| **rainfall**     | `float`                | Curah hujan tahunan di wilayah tanam, diukur dalam milimeter (mm).                               |
| **label**        | `category` / `string`  | Jenis tanaman yang direkomendasikan untuk ditanam berdasarkan parameter yang ada.                |

---

### 🚀 Workflow Benchmark
1. Import Library dan Setup Lingkungan  
2. Load Dataset 
3. Eksplorasi Data Singkat 
4. Pra-Pemrosesan Data & Split Data  
5. Benchmark Training & Evaluasi
    - KNN Model
    - Random Fosret Model
6. Ringkasan Hasil Benchmark


## 1. Import Library & Setup
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import cudf

## 2. Load Dataset
---

In [ ]:
# CPU
start_cpu = time.time()
df_cpu = pd.read_csv("synthetic_crop.csv")
cpu_load_time = time.time() - start_cpu
print(f"CPU load time: {cpu_load_time:.4f} s")
df_cpu.head()

In [ ]:
# GPU
start_gpu = time.time()
df_gpu = cudf.read_csv("synthetic_crop.csv")
gpu_load_time = time.time() - start_gpu
print(f"GPU load time: {gpu_load_time:.4f} s")
df_gpu.head()

## 3. Eksplorasi Data Singkat
---

In [ ]:
print(df_cpu.info())
print(df_cpu.describe())
print(df_cpu['label'].value_counts())

## 4. Pra-Pemrosesan Data & Split Data
---

In [ ]:
# CPU
features_cpu = df_cpu[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
target_cpu = df_cpu['label']

# Split Data CPU
from sklearn.model_selection import train_test_split
x_train_cpu, x_test_cpu, y_train_cpu, y_test_cpu = train_test_split(
    features_cpu, target_cpu, test_size=0.2, random_state=42)

In [ ]:
# GPU
from cuml.preprocessing import LabelEncoder
le = LabelEncoder()
features_gpu = df_gpu[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
target_gpu = le.fit_transform(df_gpu['label'])

# Split Data GPU
from cuml.model_selection import train_test_split as gpu_train_test_split
x_train_gpu, x_test_gpu, y_train_gpu, y_test_gpu = gpu_train_test_split(
    features_gpu, target_gpu, test_size=0.2, random_state=42)

## 5. Benchmark Training & Evaluation 
---

### KNN Model

In [ ]:
# CPU KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

start = time.time()
knn_cpu = KNeighborsClassifier()
knn_cpu.fit(x_train_cpu, y_train_cpu)
pred_cpu = knn_cpu.predict(x_test_cpu)
cpu_knn_time = time.time() - start
cpu_knn_acc = accuracy_score(y_test_cpu, pred_cpu)
print(f"CPU KNN Accuracy: {cpu_knn_acc:.4f}, Time: {cpu_knn_time:.4f} s")

In [ ]:
# GPU KNN
from cuml.neighbors import KNeighborsClassifier as cuKNN
from cuml import metrics as cuml_metrics

start = time.time()
knn_gpu = cuKNN()
knn_gpu.fit(x_train_gpu, y_train_gpu)
pred_gpu = knn_gpu.predict(x_test_gpu)
gpu_knn_time = time.time() - start
gpu_knn_acc = float(cuml_metrics.accuracy_score(y_test_gpu, pred_gpu))
print(f"GPU KNN Accuracy: {gpu_knn_acc:.4f}, Time: {gpu_knn_time:.4f} s")

### Random Forest Model

In [ ]:
# CPU Random Forest
from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf_cpu = RandomForestClassifier(n_estimators=20, random_state=0)
rf_cpu.fit(x_train_cpu, y_train_cpu)
pred_cpu = rf_cpu.predict(x_test_cpu)
cpu_rf_time = time.time() - start
cpu_rf_acc = accuracy_score(y_test_cpu, pred_cpu)
print(f"CPU Random Forest Accuracy: {cpu_rf_acc:.4f}, Time: {cpu_rf_time:.4f} s")

In [ ]:
# GPU Random Forest
from cuml.ensemble import RandomForestClassifier as cuRF

start = time.time()
rf_gpu = cuRF(n_estimators=20, random_state=0)
rf_gpu.fit(x_train_gpu, y_train_gpu)
pred_gpu = rf_gpu.predict(x_test_gpu)
gpu_rf_time = time.time() - start
gpu_rf_acc = float(cuml_metrics.accuracy_score(y_test_gpu, pred_gpu))
print(f"GPU Random Forest Accuracy: {gpu_rf_acc:.4f}, Time: {gpu_rf_time:.4f} s")

## 6. Ringkasan Hasil Benchmark 
---

In [ ]:

# Membuat DataFrame hasil
results = pd.DataFrame({
    "Model": ["KNN", "Random Forest"],
    "CPU Time (s)": [cpu_knn_time, cpu_rf_time],
    "GPU Time (s)": [gpu_knn_time, gpu_rf_time],
})

# Visualisasi waktu training
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
results.plot(x="Model", y=["CPU Time (s)", "GPU Time (s)"], kind="bar", ax=ax)
ax.set_title("Waktu Training (detik)")
ax.set_ylabel("Waktu (detik)")
plt.tight_layout()
plt.show()

## 7. Kesimpulan
---

> Notebook ini membandingkan performa training model crop recommendation antara CPU (scikit-learn) dan GPU (RAPIDS cuML) untuk KNN dan Random Forest. Hasil dapat berbeda tergantung ukuran data dan spesifikasi hardware.